# Фаза 6: подготовка папок и удалённый агент

Нужно указать свои пути и пройти ячейки по порядку. Подготовка работает без ИИ, исходники не удаляет. Осциллограммы учитываются по именам и хешам; каналы не читаются. Сетевая обработка включается отдельно. Полный словарь добавляется автоматически к каждому контексту.


(Примечание автора: А поясни чуть подробнее, чем эти пути отличаются и что туда сохраняется? Только суммарные отчёты можно ли какой-то режим, чтобы перерабатывалась исходная папка? Я там специально сделаю её уже копией исходной, и мол, если уже переработана эта папка (можно по наличию нужных файлов обработки) – то не смотреть её. И ещё раз, хотелось бы чуть более подробно описать поля тут)

In [ ]:
from pathlib import Path
import sys, json, os
from datetime import datetime

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "osc_tools/corpus").is_dir()), None)
if PROJECT is None:
    raise RuntimeError("Запустите ноутбук внутри репозитория или задайте PROJECT вручную")
sys.path.insert(0, str(PROJECT))
from osc_tools.corpus.preparation import PreparationConfig, CorpusPreparer, prepare_agents
from osc_tools.corpus.remote_agent import run_remote, run_batch
from osc_tools.corpus.record_identity import register_derivative, match_flat, restore_flat
from osc_tools.corpus.agent_runtime import write_json

# ИЗМЕНИТЬ: корень должен содержать каталоги годов.
SOURCE_ROOT = PROJECT / "Вордовские файлы, примеры"
YEARS = []  # например ["2025"]; пустой список — все годы
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = PROJECT / "private/phase6_runs" / ("prepared_" + RUN_ID)
BLOB_ROOT = PROJECT / "private/phase6_runs/blobs"  # общее хранилище нескольких запусков
SEED = PROJECT / "docs/phase_discription/phase_6/case_registry.json"  # для этой выборки; другой корпус — свой реестр/None
DOCS = PROJECT / "docs/phase_discription/phase_6"
# os.environ["PHASE6_7ZIP"] = r"C:\Program Files\7-Zip\7z.exe"  # если 7z не в PATH
WRITE_SOURCE_INDEXES = False  # True: записать два служебных JSON в исходные папки
print({"source": str(SOURCE_ROOT), "years": YEARS, "output": str(OUTPUT_ROOT)})


## Подготовка без ИИ

Проверь пути выше. Новый запуск создаёт новый выход; повтор ячейки ниже показывает уже полученную сводку. Для изменённого корня/правил выбери новый OUTPUT_ROOT. Требуются pypdf/Pillow для чтения/изображений; 7-Zip для RAR/7z/DOC и pdftoppm для страниц PDF. Этот ноутбук их сам не устанавливает.


(Примечание автора: Можно про «новый запуск – новый путь» и тут уместнее, в общем хотелось бы иметь возможность в одной папке всю подготовку делать, например, если не задан выходной путь, вот и всё.)

In [ ]:
config = PreparationConfig(
    source_root=str(SOURCE_ROOT), output_root=str(OUTPUT_ROOT),
    seed_inventory=str(SEED) if SEED else None, years=tuple(YEARS),
    seven_zip=os.environ.get("PHASE6_7ZIP"), blob_root=str(BLOB_ROOT),
    write_source_indexes=WRITE_SOURCE_INDEXES)
if (OUTPUT_ROOT / "summary.json").exists():
    from dataclasses import asdict
    saved = json.loads((OUTPUT_ROOT / "preparation_config.json").read_text(encoding="utf-8"))
    expected = asdict(config)
    expected["years"] = list(expected["years"])
    if saved != expected:
        raise ValueError("Конфигурация изменилась: выберите новый OUTPUT_ROOT")
    summary = json.loads((OUTPUT_ROOT / "summary.json").read_text(encoding="utf-8"))
else:
    summary = CorpusPreparer(config).run()
summary


In [ ]:
provenance = json.loads((OUTPUT_ROOT / "provenance_index.json").read_text(encoding="utf-8"))
archives = json.loads((OUTPUT_ROOT / "archive_manifest.json").read_text(encoding="utf-8"))
display([{"case_id": c["case_id"], "folder": c["source_folder"], "files": len(c["files"])} for c in provenance["cases"]])
display([{"archive": a["path"], "status": a["status"], "members": a.get("total_members"),
          "issues": [m for m in a["members"] if m["status"] not in {"extracted", "directory", "listed_not_read", "nested_archive_processed"}]} for a in archives])
display([{ "case_id": c["case_id"], "file": f["logical_path"], "status": f["preparation_status"] }
         for c in provenance["cases"] for f in c["files"] if not f.get("prepared_path")])


## Подготовить контексты

Для текущей демонстрационной выборки case_000005 — G06. См. reports/README.md для G ↔ case. Для своего корпуса выбери ID из таблицы выше. Для массового прохода CASE_IDS=None; повторные независимые прогоны: разные AGENT_RUN, например api_001/api_002/api_003. Возобновление того же контекста не считается независимым повтором.


In [ ]:
CASE_IDS = ["case_000005"]  # сначала один комплект; None — все подготовленные
AGENT_RUN = "api_001"
contexts = prepare_agents(OUTPUT_ROOT, DOCS / "PHASE_6_EXTRACTION_PROMPT_DRAFT.md",
    DOCS / "PHASE_6_TERMINOLOGY_DRAFT.md", case_ids=CASE_IDS, run_name=AGENT_RUN, reuse_existing=True)
labels = {c["case_id"]: c["source_folder"] for c in provenance["cases"]}
display([{**c, "source_folder": labels[c["case_id"]]} for c in contexts])


## Один удалённый API-пилот

Укажи доступную модель, задай OPENAI_API_KEY в окружении процесса Jupyter до запуска. Не записывай ключ в файл ноутбука. RUN_REMOTE=True отправляет выбранные тексты/изображения провайдеру и расходует API-токены. Цена автоматически не вычисляется. Сначала один комплект; полученный черновик ещё требует сверки по исходникам. Все ответы и usage сохраняются в рабочей папке.


In [ ]:
MODEL = ""  # точное имя доступной модели с function calling; для сканов — со зрением
RUN_REMOTE = False
RESUME = False
if RUN_REMOTE:
    if not MODEL or not os.environ.get("OPENAI_API_KEY"):
        raise ValueError("Нужны MODEL и OPENAI_API_KEY в окружении")
    result = run_remote(contexts[0]["workspace"], MODEL, max_requests=80, resume=RESUME)
    display(result)
else:
    print("Сетевой прогон выключен; подготовка контекстов не вызывает модель.")


## Пакет после проверки первого результата

Порядок — от последних годов к ранним. Завершённые контексты пропускаются. При ошибках записи остаются в журнале; это не сигнал приёмки. Для всех папок сначала CASE_IDS=None и повтор подготовки контекстов. Этот batch последователен, не распределённая очередь.


In [ ]:
RUN_BATCH = False
if RUN_BATCH:
    if not MODEL or not os.environ.get("OPENAI_API_KEY"):
        raise ValueError("Нужны MODEL и ключ в окружении")
    batch_results = run_batch([c["workspace"] for c in contexts], MODEL, resume=RESUME, max_requests=80)
    write_json(OUTPUT_ROOT.parent / (OUTPUT_ROOT.name + "__" + AGENT_RUN + "__batch.json"), batch_results)
    display(batch_results)


## Карта записи и будущих производных

Пока только учёт, без чтения сигналов. Когда отдельный обработчик создаст преобразованные файлы, зарегистрируй их происхождение сразу. Хеш исходника не позволяет узнать преобразованный файл, если эту связь не сохранить. Неоднозначный DAT с разными CFG не возвращается автоматически.


In [ ]:
RECORD_INDEX = OUTPUT_ROOT / "record_index.json"
records = json.loads(RECORD_INDEX.read_text(encoding="utf-8"))
display([{"record_id": r["record_id"], "case_id": r["case_id"], "folder": r["source_folder"],
          "kind": r["kind"], "components": [f["logical_path"] for f in r["components"]]} for r in records["records"]])
# После будущего преобразования (не запускать с примерными путями):
# register_derivative(RECORD_INDEX, record_id, [new_cfg, new_dat], "conversion",
#                     {"crop": False, "processor_version": "..."})


## Безымянный набор: сначала план, затем копирование

Укажи FLAT_ROOT, посмотри matches/status. UNKNOWN требует карты преобразования/ручной сверки; MULTIPLE_MATCHES_REVIEW — неоднозначный хеш. RESTORE_ROOT должен быть новым и вне первичного корня. Возврат копирует, исходные файлы не перемещает.


In [ ]:
FLAT_ROOT = None  # Path(r"D:\ваш_безымянный_набор")
RESTORE_ROOT = None  # Path(r"D:\новый_каталог_возврата")
APPLY_RESTORE = False
if FLAT_ROOT is not None:
    return_plan = match_flat(RECORD_INDEX, FLAT_ROOT)
    write_json(OUTPUT_ROOT / "flat_return_plan.json", return_plan)
    display(return_plan["files"])
    if APPLY_RESTORE:
        if RESTORE_ROOT is None:
            raise ValueError("Укажите новый RESTORE_ROOT")
        display(restore_flat(return_plan, RESTORE_ROOT))


## Что сверить человеку

В reports/README.md — все 17 черновиков и конкретные вопросы G03/D020/G07/G12. Проверка текста без первичных документов не является приёмкой. После API-пилота сравним факты/пропуски/оценки/редакции, затем при ресурсах сделаем три независимых прогона. Дальнее изучение структуры/статистики отчётов записано в плане и сейчас не выполняется.
